In [0]:
from pyspark.sql import functions as F
df_silver = spark.read.table("restaurant_project.silver.rest_sales")

## Check for NULLs or Duplicates in sales_skey

In [0]:
print('>> Checking sales_skey: NULLs & Duplicates')

pk_issues = df_silver.groupby("sales_skey").count().filter("COUNT > 1 OR sales_skey IS NULL")

if pk_issues.isEmpty():
    print("✅ Pass: No duplicates or NULLs in sales_skey.")
else:
    print(f"❌ Fail: Found issues in sales_skey.")
    display(pk_issues)




## Check for NULLs in Essential Columns

In [0]:
essential_cols = ["item_name", "price", "quantity", "order_date"]

for col_name in essential_cols:
    null_count = df_silver.filter(F.col(col_name).isNull()).count()
    if null_count == 0:
        print(f"✅ Pass: No NULLs in {col_name}.")
    else:
        print(f"❌ Fail: Found {null_count} NULLs in {col_name}.")

## Check for Data Integrity

In [0]:
print('>> Checking Business Logic: Price & Discount')
logic_issues = df_silver.filter("price <= 0 OR discount > price")

if logic_issues.isEmpty():
     print("✅ Pass: All records satisfy business rules (price > 0 and discount <= price).")
else:
    print(f"❌ Fail: Found {logic_issues.count()} invalid records.")
    display(logic_issues.limit(10))

## Check for Trim Issues (Whitespace Check)

In [0]:
whitespace_issues = df_silver.filter(
    (F.col("item_name") != F.trim(F.col("item_name")))|
    (F.col("category") != F.trim(F.col("category")))|
    (F.col("branch") != F.trim(F.col("branch")))|
    (F.col("payment_method") != F.trim(F.col("payment_method")))|
    (F.col("order_type") != F.trim(F.col("order_type")))
)

if whitespace_issues.isEmpty():
    print("✅ Pass: No leading/trailing whitespaces found in text columns.")
else:
    print(f"❌ Fail: Found {whitespace_issues.count()} records with whitespace issues.")
    display(whitespace_issues.select("item_name", "category").limit(10))

## Total Amount Check

In [0]:
print('>> Checking Total Amount Calculation Accuracy')

calculation_issues = df_silver.withColumn(
    "calc_check", 
    F.abs(F.col("total_amount") - F.round((F.col("price") * F.col("quantity")) * (F.lit(1) - F.col("discount")), 2))
).filter("calc_check > 0.01").count()

if calculation_issues == 0:
    print("✅ Pass: All total_amount calculations are accurate.")
else:
    print(f"❌ Fail: Found {calculation_issues} rows with calculation mismatch.")